In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!rm -rf PortfolioFlow-AI
!git clone https://github.com/RajSarangam/PortfolioFlow-AI.git
%cd PortfolioFlow-AI

Cloning into 'PortfolioFlow-AI'...
remote: Enumerating objects: 67, done.
remote: Counting objects: 100% (67/67), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 67 (delta 9), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (67/67), 37.24 KiB | 1.55 MiB/s, done.
Resolving deltas: 100% (9/9), done.
/kaggle/working/PortfolioFlow-AI


In [3]:
%cd /kaggle/working
!rm -rf PortfolioFlow-AI
!git clone https://github.com/RajSarangam/PortfolioFlow-AI.git
%cd /kaggle/working/PortfolioFlow-AI
import os
print("cwd:", os.getcwd())
print("files:", os.listdir())

/kaggle/working
Cloning into 'PortfolioFlow-AI'...
remote: Enumerating objects: 67, done.
remote: Counting objects: 100% (67/67), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 67 (delta 9), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (67/67), 37.24 KiB | 12.41 MiB/s, done.
Resolving deltas: 100% (9/9), done.
/kaggle/working/PortfolioFlow-AI
cwd: /kaggle/working/PortfolioFlow-AI
files: ['portfolio-ai.ipynb', 'requirements.txt', 'WRITEUP.md', 'LICENSE', '.gitignore', 'mcp_server', 'notebooks', 'intake_triage', 'data', '.git', 'README.md']


In [4]:
!pip -q install google-adk google-genai pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 306.0/306.0 kB 6.0 MB/s eta 0:00:00a 0:00:01


In [5]:
from kaggle_secrets import UserSecretsClient
import os
os.environ['GOOGLE_API_KEY'] = UserSecretsClient().get_secret('GOOGLE_API_KEY')
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'FALSE'
print('Gemini key loaded:', bool(os.environ.get('GOOGLE_API_KEY')))

Gemini key loaded: True


In [9]:
import os, sys

# --- make the repo importable regardless of cwd ---
REPO = "/kaggle/working/PortfolioFlow-AI"
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
assert os.path.isdir(f"{REPO}/intake_triage"), "intake_triage not found — re-run the clone cell first"

# --- build the runner ---
import warnings, json, re, itertools
warnings.filterwarnings('ignore')

from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types as gt
from intake_triage import root_agent

session_service = InMemorySessionService()
runner = Runner(agent=root_agent, app_name='intake_triage', session_service=session_service)
_counter = itertools.count(1)

def _safe_json(text):
    """Parse a JSON string that may be wrapped in code fences."""
    if not text:
        return None
    cleaned = re.sub(r'^```(json)?|```$', '', text.strip(), flags=re.MULTILINE).strip()
    try:
        return json.loads(cleaned)
    except Exception:
        return {'_raw': text}

def triage(request_text):
    """Run the parse -> score -> route pipeline and return the final session state."""
    uid, sid = "pm", f"s-{next(_counter)}"          # unique session per call
    session_service.create_session_sync(app_name='intake_triage',
                                        user_id=uid, session_id=sid, state={})
    msg = gt.Content(role='user', parts=[gt.Part.from_text(text=request_text)])
    for _ in runner.run(user_id=uid, session_id=sid, new_message=msg):
        pass                                         # drive all three agents to completion
    s = session_service.get_session_sync(app_name='intake_triage',
                                         user_id=uid, session_id=sid)
    return s.state

print('runner ready')

runner ready


In [10]:
SAMPLE = (
    'Hi team, Sales wants a single view of each customer pulling together Salesforce '
    'records, support tickets, and product usage so reps can see churn risk. We think '
    'this is important and would like it within the quarter. Not sure who owns it.'
)

state = triage(SAMPLE)
parsed     = _safe_json(state.get('parsed_request'))
assessment = _safe_json(state.get('triage_assessment'))
routing    = _safe_json(state.get('routing_recommendation'))

print('PARSED:');       print(json.dumps(parsed, indent=2))
print('\nASSESSMENT:'); print(json.dumps(assessment, indent=2))
print('\nROUTING:');    print(json.dumps(routing, indent=2))

Deprecated. Please migrate to the async method.


[guardrail] tool call: find_related_projects args={'keywords': 'customer view, churn risk, Salesforce integration, support tickets, product usage, sales enablement, data integration'}
[guardrail] tool call: lookup_capacity args={'team_name': 'CRM & Revenue'}


Deprecated. Please migrate to the async method.


PARSED:
{
  "requestor": null,
  "business_unit": "Sales",
  "problem_statement": "Sales representatives lack a unified view of customer data (Salesforce records, support tickets, product usage) to effectively assess churn risk.",
  "desired_outcome": "A single, comprehensive view of each customer, integrating data from Salesforce, support tickets, and product usage, to enable sales reps to identify churn risk.",
  "urgency": "high",
  "rough_size": "large",
  "dependencies": [],
  "keywords": [
    "customer view",
    "churn risk",
    "Salesforce integration",
    "support tickets",
    "product usage",
    "sales enablement",
    "data integration"
  ],
  "missing_info": [
    "Project owner",
    "Specific data sources/systems",
    "Detailed scope requirements",
    "Current process for identifying churn risk",
    "Required fields for the customer view"
  ]
}

ASSESSMENT:
{
  "scores": {
    "value": 5,
    "effort": 5,
    "risk": 4,
    "strategic_alignment": 3
  },
  "rationa

In [12]:
from intake_triage.skills.triage_rubric import score_to_tier
from intake_triage.guardrails import screen_request, enforce_human_in_the_loop

# pass positionally so the parameter name doesn't matter
if assessment and 'scores' in assessment:
    s = assessment['scores']
    tier = score_to_tier(s['value'], s['effort'], s['risk'], s['strategic_alignment'])
else:
    tier = {}

flags = []
flags += screen_request(SAMPLE)['input_flags']
flags += (assessment or {}).get('missing_info', [])
flags += (routing or {}).get('open_questions', [])

final = enforce_human_in_the_loop(routing or {}, flags)
final['priority'] = tier

print(json.dumps(final, indent=2))

{
  "recommendation": "needs_more_info",
  "decision_authority": "human",
  "auto_action_taken": false,
  "human_review_required": true,
  "confidence": "low",
  "review_flags": [
    "Project owner",
    "Specific data sources/systems",
    "Detailed scope requirements",
    "Current process for identifying churn risk",
    "Required fields for the customer view",
    "Strategic alignment criteria",
    "How does this request relate to the existing 'Customer Health Scoring' (PRJ-1009) and 'Unified Customer 360' (PRJ-1001) projects within the CRM & Revenue team? Is this a new ask, an extension, or a potential duplicate?",
    "Who is the designated project owner for this initiative?",
    "Can you provide specific data sources/systems (e.g., specific Salesforce clouds, support ticketing system name, product usage analytics tool)?",
    "Can you elaborate on detailed scope requirements, including the specific fields needed for the customer view and any desired UI/UX considerations?",
  